In [1]:
import os
import pandas as pd

results_path = '../../data/results/moabb/erp'

df = []
for root, dirs, files in os.walk(results_path):
    for name in files:
        if name.endswith(".csv"):
            path = os.path.join(root, name)
            df.append(pd.read_csv(path))
df = pd.concat(df, ignore_index=True)
df

,Unnamed: 0,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0,0.903495,14.182683,768.0,1,0,16,1,BrainInvaders2012,HODA
1,1,0.951022,60.850376,768.0,1,0,16,1,BrainInvaders2012,PARAFACDA
2,2,0.943053,568.618350,768.0,1,0,16,1,BrainInvaders2012,BTTDA
3,0,0.920676,16.953000,764.0,2,0,16,1,BrainInvaders2012,HODA
4,1,0.948905,63.900970,764.0,2,0,16,1,BrainInvaders2012,PARAFACDA
...,...,...,...,...,...,...,...,...,...,...
895,1,0.790365,675.732540,320.0,29,0,30,1,ErpCore2021-N170,BTTDA
896,2,0.818490,20.648365,320.0,29,0,30,1,ErpCore2021-N170,HODA
897,0,0.976302,80.188934,320.0,4,0,30,1,ErpCore2021-N170,PARAFACDA
898,1,0.971615,868.946700,320.0,4,0,30,1,ErpCore2021-N170,BTTDA


In [2]:
subj_count = df.groupby('dataset')['subject'].nunique()
subj_count

dataset
BNCI2014-008           8
BNCI2015-003          10
BrainInvaders2012     25
BrainInvaders2014a    64
BrainInvaders2014b    38
BrainInvaders2015b    44
Cattan2019-VR         21
ErpCore2021-ERN       28
ErpCore2021-MMN       34
ErpCore2021-N170      28
Name: subject, dtype: int64

In [3]:
subj_count = df.groupby('dataset')['subject'].unique()['BrainInvaders2015b']
sorted(subj_count);

In [4]:
score = df.groupby(['dataset', 'pipeline']).score.aggregate(['mean', 'std'])
score = (score*100).round(2)
score

mean    std
dataset            pipeline               
BNCI2014-008       BTTDA      86.06   4.62
                   HODA       84.73   5.35
                   PARAFACDA  86.05   4.66
BNCI2015-003       BTTDA      84.87   7.67
                   HODA       82.93   7.37
                   PARAFACDA  84.52   7.74
BrainInvaders2012  BTTDA      90.88   4.94
                   HODA       87.34   5.58
                   PARAFACDA  90.65   5.19
BrainInvaders2014a BTTDA      87.98   9.10
                   HODA       84.15  10.32
                   PARAFACDA  87.29   9.61
BrainInvaders2014b BTTDA      90.65  10.91
                   HODA       87.55  10.14
                   PARAFACDA  89.56   9.33
BrainInvaders2015b BTTDA      84.51  12.17
                   HODA       83.20  12.62
                   PARAFACDA  84.37  12.31
Cattan2019-VR      BTTDA      91.34   7.67
                   HODA       88.84   8.16
                   PARAFACDA  90.75   8.17
ErpCore2021-ERN    BTTDA      94.78   3.95
                   HODA       92.74   8.64
                   PARAFACDA  93.00   7.29
ErpCore2021-MMN    BTTDA      63.53   7.90
                   HODA       63.79   7.45
                   PARAFACDA  63.36   7.39
ErpCore2021-N170   BTTDA      91.42   7.99
                   HODA       89.13   8.75
                   PARAFACDA  91.15   7.79

In [5]:
df_diff = df.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

pipeline,subject,session,channels,n_sessions,samples,dataset,BTTDA,HODA,PARAFACDA,score_diff
0,1,0,8,1,4200.0,BNCI2014-008,0.831961,0.809818,0.826312,0.022143
1,1,0,8,1,5400.0,BNCI2015-003,0.789784,0.760889,0.788838,0.028895
2,1,0,16,1,768.0,BrainInvaders2012,0.943053,0.903495,0.951022,0.039558
3,1,0,16,1,1188.0,BrainInvaders2014a,0.955743,0.917372,0.947960,0.038372
4,1,0,30,1,320.0,ErpCore2021-N170,0.841146,0.788542,0.815104,0.052604
...,...,...,...,...,...,...,...,...,...,...
295,60,0,16,1,792.0,BrainInvaders2014a,0.968333,0.967554,0.974791,0.000779
296,61,0,16,1,1368.0,BrainInvaders2014a,0.776999,0.684261,0.779355,0.092739
297,62,0,16,1,1367.0,BrainInvaders2014a,0.565078,0.553237,0.544812,0.011841
298,63,0,16,1,420.0,BrainInvaders2014a,0.648571,0.595306,0.627347,0.053265


In [93]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[.5, 1])
    fig.update_yaxes(range=[.5, 1])
    fig.add_shape(
        type="line",
        x0=0.5, y0=0.5, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

In [94]:
fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1400,
    height=600,
)
fig.update_layout(showlegend=False)
fig

In [8]:
compare_score_plot(df_diff, 'HODA', 'PARAFACDA')

In [9]:
compare_score_plot(df_diff, 'PARAFACDA', 'BTTDA')

In [10]:


fig = px.scatter_matrix(
    df_diff,
    dimensions=['channels', 'samples', 'BTTDA', 'HODA', 'PARAFACDA', 'score_diff'],
    color='dataset',
)
fig.update_traces(diagonal_visible=False, showupperhalf=False)
fig.update_layout(
    width=2000,
    height=1800,
)
fig


		mean 	std
dataset 	pipeline 		
BrainInvaders2012 	BTTDA 	90.64 	4.96
HODA 	88.25 	4.98
PARAFACDA 	90.50 	4.87


before: old
to try:
    proper zscoring -> helped a little (old2)
    refit_shrinkage=True -> helped a little (old3)
    toeplitz -> helped a little (old4)
    SelectFdr (which cutoff?) helped a little
    rt instead of tr
    kNN/SVM

##### results